# Chapter 12 — DDPM: Forward and Reverse Diffusion
**Math for ML Engineers** · companion notebook

This notebook implements the DDPM forward and reverse processes from §12.6 using a closed-form oracle denoiser so you can see the mathematics without training.

**Equations:** forward marginal (12.37), reverse-step mean (12.47), noise schedule $\bar{\alpha}_t$ (Forward Diffusion Process, around (12.36)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Noise schedule

DDPM defines $T = 1000$ forward steps.  At each step $t$, Gaussian noise is added according to variance $\beta_t$ (linearly spaced from $\beta_{\min}$ to $\beta_{\max}$):

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^t \alpha_s$$

The **closed-form forward marginal** is:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\ (1 - \bar{\alpha}_t)\,\mathbf{I})$$

This lets us jump to any noise level $t$ in one step — no need to iterate.

In [ ]:
T = 1000
beta_min, beta_max = 1e-4, 0.02
betas = np.linspace(beta_min, beta_max, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(betas); plt.xlabel('Step $t$'); plt.ylabel(r'$\beta_t$')
plt.title('Noise schedule $\\beta_t$')
plt.subplot(1, 2, 2)
plt.plot(alpha_bar, color='C1'); plt.xlabel('Step $t$'); plt.ylabel(r'$\bar{\alpha}_t$')
plt.title('Signal retention $\\bar{\\alpha}_t$')
plt.tight_layout(); plt.show()
print(f'Signal at t=250: {alpha_bar[249]:.3f}   t=500: {alpha_bar[499]:.3f}   t=999: {alpha_bar[999]:.4f}')


## Forward process: adding noise

For a 1-D signal $x_0 = 1.0$, we visualise how the distribution $q(x_t \mid x_0)$ widens as $t$ increases.  By $t \approx 700$ the signal is essentially indistinguishable from $\mathcal{N}(0,1)$.

In [ ]:
rng = np.random.default_rng(42)
x0 = 1.0
steps_to_show = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(steps_to_show), figsize=(14, 2.5), sharey=True)
for ax, t in zip(axes, steps_to_show):
    ab = alpha_bar[t] if t > 0 else 1.0
    # Sample 5000 points from q(x_t | x_0)
    samples = np.sqrt(ab)*x0 + np.sqrt(1-ab)*rng.standard_normal(5000)
    ax.hist(samples, bins=40, density=True, color='C0', alpha=0.7)
    ax.axvline(np.sqrt(ab)*x0, color='C3', lw=1.5, label=f'mean={np.sqrt(ab)*x0:.2f}')
    ax.set_title(f't = {t}\n$\\bar\\alpha$={ab:.3f}')
    ax.set_xlabel('$x_t$')
axes[0].set_ylabel('Density')
plt.suptitle('Forward diffusion: $q(x_t | x_0=1)$ at increasing noise levels', y=1.05)
plt.tight_layout(); plt.show()


## Reverse process: denoising

The reverse step uses Bayes' rule to derive:

$$q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}(\tilde{\mu}_t(\mathbf{x}_t, \mathbf{x}_0),\ \tilde{\beta}_t \mathbf{I})$$

$$\tilde{\mu}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t}\,\mathbf{x}_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}\,\mathbf{x}_t, \qquad \tilde{\beta}_t = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\,\beta_t$$

Here we use the **oracle** denoiser (plug in the true $x_0$).  A trained DDPM network learns to predict $x_0$ (or equivalently $\epsilon$) from $(x_t, t)$.

In [ ]:
def reverse_step_oracle(xt, x0, t, rng):
    '''One reverse DDPM step using the oracle denoiser (true x0 known).'''
    if t == 0:
        return x0
    ab_t   = alpha_bar[t]
    ab_tm1 = alpha_bar[t-1]
    beta_t = betas[t]
    alpha_t = alphas[t]
    # Posterior mean
    mu = (np.sqrt(ab_tm1)*beta_t / (1-ab_t))*x0 + \
         (np.sqrt(alpha_t)*(1-ab_tm1) / (1-ab_t))*xt
    # Posterior variance
    var = (1-ab_tm1)/(1-ab_t) * beta_t
    return mu + np.sqrt(var)*rng.standard_normal(xt.shape)

# Full forward + reverse on 1-D data, n_particles = 500
n_particles = 500
x0_samples = rng.standard_normal(n_particles) * 0.3 + 1.0  # cluster near 1

# Forward to T
xt_final = np.sqrt(alpha_bar[-1])*x0_samples + np.sqrt(1-alpha_bar[-1])*rng.standard_normal(n_particles)

# Reverse all the way back
xt = xt_final.copy()
snapshots = {999: xt.copy()}
for t in range(T-1, -1, -1):  # include t=0 so the oracle base case (return x0) actually fires
    xt = reverse_step_oracle(xt, x0_samples, t, rng)
    if t in [750, 500, 250, 100, 0]:
        snapshots[t] = xt.copy()

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5), sharey=True)
for ax, t in zip(axes, [999, 750, 500, 250, 100, 0]):
    ax.hist(snapshots[t], bins=30, density=True, color='C1', alpha=0.7)
    ax.set_title(f't={t}')
    ax.set_xlabel('$x_t$')
axes[0].set_ylabel('Density')
plt.suptitle('Reverse diffusion: samples converge back to data distribution', y=1.05)
plt.tight_layout(); plt.show()


## DDIM: deterministic sampling in 50 steps

DDIM replaces the stochastic reverse step with a deterministic ODE integration, allowing 50-step inference (vs 1 000 for DDPM) with negligible quality loss.  This is a $20\times$ speedup — a critical production number (reference card, §Generative Models).

In [ ]:
# DDIM with 50 steps (uniform stride)
ddim_steps = 50
stride = T // ddim_steps
timesteps = list(range(0, T, stride))[::-1]

def ddim_step_oracle(xt, x0, t, t_prev):
    '''DDIM deterministic reverse step.'''
    ab_t    = alpha_bar[t]
    ab_prev = alpha_bar[t_prev] if t_prev >= 0 else 1.0
    # Predicted x0 (oracle)
    eps = (xt - np.sqrt(ab_t)*x0) / np.sqrt(1-ab_t)
    return np.sqrt(ab_prev)*x0 + np.sqrt(1-ab_prev)*eps

xt_ddim = xt_final.copy()
for i in range(len(timesteps)-1):
    t, t_prev = timesteps[i], timesteps[i+1]
    xt_ddim = ddim_step_oracle(xt_ddim, x0_samples, t, t_prev)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].hist(x0_samples, bins=30, density=True, color='C0', alpha=0.7); axes[0].set_title('True $x_0$')
axes[1].hist(snapshots[0], bins=30, density=True, color='C1', alpha=0.7); axes[1].set_title('DDPM (1000 steps)')
axes[2].hist(xt_ddim, bins=30, density=True, color='C2', alpha=0.7); axes[2].set_title('DDIM (50 steps)')
for ax in axes: ax.set_xlabel('x')
plt.suptitle('DDPM vs DDIM — same quality, 20× fewer steps', y=1.02)
plt.tight_layout(); plt.show()


## Exercise

1. Change `x0_samples` to a bimodal distribution (two clusters at $-1$ and $+1$) and re-run the reverse process.  Does DDPM recover both modes?  At what step $t$ do the two clusters become distinguishable?
2. Run DDIM with `ddim_steps = 8` (flow-matching territory).  How does the recovered distribution compare?  What does this tell you about why flow matching uses straight paths (§12.6.10)?